# Official Mamba backbone on Colab

This notebook runs the official `mamba-ssm` sequence backbone on the four-dataset battery RUL benchmark.

- Backend: official `mamba-ssm` (`from mamba_ssm import Mamba`)
- Same sequence contract, splits, metrics, source-range clipping, checkpointing, and 9-config full grid as the CNN/Transformer runners.

Use a Colab GPU runtime (`Runtime > Change runtime type > T4 GPU`). Outputs/checkpoints are written to Google Drive so the run can resume after a disconnect.

In [ ]:
# 1. Select run mode.
# Recommended runtime: Colab T4 GPU.
RUN_SMOKE_FIRST = True
RUN_FULL_BENCHMARK = True
FORCE_RERUN_FULL = False  # keep False to resume from Drive checkpoints after Colab disconnects

REPO_URL = "https://github.com/osmansafacifci/Graduation-Project-Dicle.git"
REPO_DIR = "/content/Graduation-Project-Dicle"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/Dicle_Mamba_Outputs"

In [ ]:
# 2. Mount Drive, clone/pull repo, and inspect GPU.
from pathlib import Path
import os, subprocess, textwrap

from google.colab import drive
drive.mount('/content/drive')

if Path(REPO_DIR).exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('repo:', Path.cwd())
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# 3. Install official Mamba.
# Mamba's official README recommends --no-build-isolation so pip uses Colab's CUDA-enabled PyTorch.
import subprocess, sys

def run(cmd, *, check=True):
    print('+', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

run([sys.executable, '-m', 'pip', 'install', '-q', 'scienceplots'], check=False)

result = run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'mamba-ssm[causal-conv1d]', '--no-build-isolation'
], check=False)
if result.returncode != 0:
    print('Primary Mamba install failed; trying a pinned fallback often used on Colab images.')
    run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'causal-conv1d==1.5.0.post8', 'mamba-ssm==2.2.4', '--no-build-isolation'
    ], check=True)
from mamba_ssm import Mamba
print('mamba-ssm import OK:', Mamba)

In [ ]:
# 4. Smoke test: one within-dataset unit. This confirms install, CUDA, data loading, and checkpoints.
from pathlib import Path
import os, subprocess

smoke_dir = Path(DRIVE_OUTPUT_ROOT) / 'results_v2_four_dataset_mamba_library_smoke'
smoke_dir.mkdir(parents=True, exist_ok=True)

if RUN_SMOKE_FIRST:
    cmd = [
        'python', '-u', '2_models/run_mamba_library_pytorch.py',
        '--hp-grid', 'quick',
        '--device', 'cuda',
        '--within-only',
        '--datasets', 'matr',
        '--seeds', '42',
        '--windows', '100',
        '--output-dir', str(smoke_dir),
        '--force-rerun',
    ]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping smoke test.')

In [ ]:
# 5. Full 80-unit benchmark with the 9-config full grid.
# This writes checkpoints to Drive and resumes if rerun with FORCE_RERUN_FULL=False.
from pathlib import Path
import os, subprocess

out_dir = Path(DRIVE_OUTPUT_ROOT) / 'results_v2_four_dataset_mamba_library_pytorch'
out_dir.mkdir(parents=True, exist_ok=True)

if RUN_FULL_BENCHMARK:
    cmd = [
        'python', '-u', '2_models/run_mamba_library_pytorch.py',
        '--hp-grid', 'full',
        '--device', 'cuda',
        '--output-dir', str(out_dir),
    ]
    if FORCE_RERUN_FULL:
        cmd.append('--force-rerun')
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping full benchmark. Set RUN_FULL_BENCHMARK=True to run it.')

print('Output directory:', out_dir)

In [ ]:
# 6. Inspect the final summary and make a zip for local download if desired.
from pathlib import Path
import pandas as pd, shutil

summary_path = Path(DRIVE_OUTPUT_ROOT) / 'results_v2_four_dataset_mamba_library_pytorch' / 'results_summary.csv'
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary[['scenario', 'experiment', 'MAE_mean', 'SMAPE_mean', 'R2_mean', 'n_runs']])
    archive_base = Path(DRIVE_OUTPUT_ROOT) / 'results_v2_four_dataset_mamba_library_pytorch'
    zip_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(archive_base))
    print('Zip written:', zip_path)
else:
    print('No full-run summary yet:', summary_path)